# memopro CUDA check, run 6: `check` accuracy (docs/research/0055)

Runs 4 and 5 missed the pre-registered ±15% (0054, 0055). Run 5 also showed that its own
measurements leaked memory from one case into the next. This run repeats **the same 6 cases
with the same criterion** after three fixes to `check` (trace the steady-state second step, keep
only the loss, use the optimizer implementation torch picks on the device), and measures **each
case in a fresh process**:

- `measured`: CUDA `max_memory_allocated` of a steady-state step, per phase (forward, backward,
  optimizer step)
- `tracker_real`: torch's MemTracker on real tensors for the same step (tracker vs allocator)
- `predicted`: `memopro.check` (fake tensors, no allocation) and its breakdown

1. **Runtime -> Change runtime type -> T4 GPU**; upload the wheel sent with run 5 to `/content`.
2. **Runtime -> Run all** (restart the session first). About 8 minutes. The first output
   must show `"check_0055": true`.
3. Copy the JSON printed by the last cell back to the conversation.

In [ ]:
import glob, json, os, subprocess, sys, time, traceback

wheels = sorted(glob.glob("/content/memopro-*.whl"))
if wheels:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                    wheels[-1]], check=True)
import memopro, torch, transformers  # noqa: E401, E402
import memopro.access._check  # noqa: E402

results = {"memopro": memopro.__version__, "python": sys.version.split()[0],
           "torch": torch.__version__, "transformers": transformers.__version__,
           "wheel": os.path.basename(wheels[-1]) if wheels else None,
           "check_0055": "keep only the loss" in open(memopro.access._check.__file__).read(),
           "cuda": torch.cuda.is_available(),
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
print(results)

In [ ]:
SCRIPT = r"""
import json, sys, torch
from transformers import AutoModelForCausalLM
from torch.distributed._tools.mem_tracker import MemTracker
from torch.optim.optimizer import _default_to_fused_or_foreach

name, goal, batch, seq = sys.argv[1], sys.argv[2], int(sys.argv[3]), int(sys.argv[4])


def peak():
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated()


def reset():
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()


def totals(snapshot):
    out = {}
    for per_device in snapshot.values():
        for key, value in per_device.items():
            k = str(key).rsplit(".", 1)[-1]
            out[k] = out.get(k, 0) + int(value)
    return out


model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).cuda()
x = torch.randint(0, model.config.vocab_size, (batch, seq), device="cuda")
out = {"allocated_after_load": torch.cuda.memory_allocated()}
try:
    if goal == "infer":
        model.eval()
        with torch.no_grad():
            model(x)
            reset()
            model(x)
            out["peak"] = peak()
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=1e-5)
        model.train()
        model(x, labels=x).loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
        phases = {}
        reset(); loss = model(x, labels=x).loss; phases["forward"] = peak()
        reset(); loss.backward(); del loss; phases["backward"] = peak()
        reset(); opt.step(); phases["step"] = peak()
        opt.zero_grad(set_to_none=True)
        out.update(phases=phases, peak=max(phases.values()), peak_phase=max(phases, key=phases.get))
        tracker = MemTracker()
        tracker.track_external(model, opt)
        with tracker:
            model(x, labels=x).loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
        out["tracker_real"] = totals(tracker.get_tracker_snapshot("peak"))
        fused, foreach = _default_to_fused_or_foreach(list(model.parameters()), False, False)
        out["optimizer_impl"] = {"foreach": foreach, "fused": fused}
except torch.OutOfMemoryError:
    out["peak"] = "oom"
print("RESULT " + json.dumps(out))
"""
with open("/content/memopro_measure.py" if os.path.isdir("/content") else "memopro_measure.py", "w") as f:
    f.write(SCRIPT)
MEASURE_PATH = f.name


def measure(name, goal, batch, seq):
    """One case in a fresh process: nothing from another case can stay allocated."""
    done = subprocess.run([sys.executable, MEASURE_PATH, name, goal, str(batch), str(seq)],
                          capture_output=True, text=True, timeout=1800, check=False)
    line = next((l for l in done.stdout.splitlines() if l.startswith("RESULT ")), None)
    if line is None:
        return {"peak": "error", "stderr": done.stderr[-800:]}
    return json.loads(line[len("RESULT "):])

In [ ]:
cases = [("openai-community/gpt2", "infer", 1, 1024), ("openai-community/gpt2", "train", 2, 512),
         ("openai-community/gpt2", "train", 8, 512),
         ("openai-community/gpt2-medium", "infer", 1, 1024),
         ("openai-community/gpt2-medium", "train", 1, 512),
         ("openai-community/gpt2-medium", "train", 2, 512)]
rows = []
for name, goal, batch, seq in cases:
    t = time.time()
    try:
        r = memopro.check(name, goal=goal, batch_size=batch, seq_len=seq, device="cuda")
        part = r.inference if goal == "infer" else r.training
        pred = part.get("peak")
        breakdown = {k: part.get(k) for k in ("params", "grads", "optimizer_state", "activations")
                     if k in part}
        real = measure(name, goal, batch, seq)
        err = None if not isinstance(real["peak"], int) or not pred else round(pred / real["peak"] - 1, 4)
        rows.append({"model": name.split("/")[-1], "goal": goal, "batch": batch, "seq": seq,
                     "predicted": pred, "predicted_breakdown": breakdown, "measured": real,
                     "rel_error": err, "within_15pct": err is not None and abs(err) <= 0.15,
                     "notes": r.notes})
    except Exception as e:
        rows.append({"model": name, "goal": goal, "batch": batch,
                     "error": f"{type(e).__name__}: {e}"[:300], "trace": traceback.format_exc()[-600:]})
    print(rows[-1].get("model"), goal, batch, f"({time.time() - t:.0f}s)", rows[-1].get("rel_error"))
measured_rows = [r for r in rows if isinstance(r.get("measured"), dict) and isinstance(r["measured"]["peak"], int)]
results["check_accuracy"] = {"cases": rows, "measured_cases": len(measured_rows),
                             "all_within_15pct": bool(measured_rows) and all(r["within_15pct"] for r in measured_rows)}

In [ ]:
print(json.dumps(results, indent=2, default=str))